In [ ]:
# ===============================
# 1. IMPORT THƯ VIỆN
# ===============================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# ===============================
# 2. ĐỌC DỮ LIỆU
# ===============================

df = pd.read_csv("FPT.csv")

print(df.columns)
df.head()
# ===============================
# 3. ĐỔI TÊN CỘT
# ===============================

df = df.rename(columns={
    "Ngày": "Date",
    "Lần cuối": "Close",
    "Mở": "Open",
    "Cao": "High",
    "Thấp": "Low",
    "KL": "Volume",
    "% Thay đổi": "Change_Percent"
})

df.head()
# ===============================
# 4. XỬ LÝ CỘT KHỐI LƯỢNG
# ===============================

def convert_volume(x):
    x = str(x).replace(",", "").strip()
    if "M" in x:
        return float(x.replace("M", "")) * 1_000_000
    elif "K" in x:
        return float(x.replace("K", "")) * 1_000
    else:
        return float(x)

df["Volume"] = df["Volume"].apply(convert_volume)

# ===============================
# 5. TIỀN XỬ LÝ DỮ LIỆU
# ===============================

df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)

df = df.sort_values("Date")

df = df[(df["Date"] >= "2020-01-01") & (df["Date"] <= "2025-12-31")]

df = df.dropna()

df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Quarter"] = df["Date"].dt.quarter

df["Daily_Change"] = df["Close"] - df["Open"]
df["Daily_Return_%"] = df["Close"].pct_change() * 100

df.head()
# ===============================
# 6. KIỂM TRA DỮ LIỆU SAU XỬ LÝ
# ===============================

print(df.info())
print(df.describe())
print(df.isnull().sum())
# Câu 1: Giá cổ phiếu FPT biến động như thế nào giai đoạn 2020–2025?

plt.figure(figsize=(14,6))
plt.plot(df["Date"], df["Close"])
plt.title("Biến động giá đóng cửa cổ phiếu FPT giai đoạn 2020-2025")
plt.xlabel("Thời gian")
plt.ylabel("Giá đóng cửa")
plt.grid(True)
plt.show()
# Câu 2: Năm nào cổ phiếu FPT có mức giá trung bình cao nhất?

gia_tb_nam = df.groupby("Year")["Close"].mean()

print(gia_tb_nam)

plt.figure(figsize=(10,5))
gia_tb_nam.plot(kind="bar")
plt.title("Giá đóng cửa trung bình theo năm")
plt.xlabel("Năm")
plt.ylabel("Giá trung bình")
plt.grid(axis="y")
plt.show()

print("Năm có giá trung bình cao nhất:", gia_tb_nam.idxmax())
print("Giá trung bình cao nhất:", gia_tb_nam.max())
# Câu 3: Thời điểm nào giá FPT cao nhất và thấp nhất?

gia_cao_nhat = df.loc[df["Close"].idxmax()]
gia_thap_nhat = df.loc[df["Close"].idxmin()]

print("Ngày có giá đóng cửa cao nhất:")
print(gia_cao_nhat[["Date", "Close", "Volume"]])

print("\nNgày có giá đóng cửa thấp nhất:")
print(gia_thap_nhat[["Date", "Close", "Volume"]])
# Câu 4: Khối lượng giao dịch có liên quan đến giá cổ phiếu không?

plt.figure(figsize=(8,6))
plt.scatter(df["Volume"], df["Close"])
plt.title("Mối quan hệ giữa khối lượng giao dịch và giá đóng cửa")
plt.xlabel("Khối lượng giao dịch")
plt.ylabel("Giá đóng cửa")
plt.grid(True)
plt.show()

correlation = df["Volume"].corr(df["Close"])
print("Hệ số tương quan giữa Volume và Close:", correlation)
# Câu 5: Giá cổ phiếu FPT biến động theo tháng như thế nào?

gia_tb_thang = df.groupby("Month")["Close"].mean()

print(gia_tb_thang)

plt.figure(figsize=(10,5))
gia_tb_thang.plot(kind="bar")
plt.title("Giá đóng cửa trung bình theo tháng")
plt.xlabel("Tháng")
plt.ylabel("Giá trung bình")
plt.grid(axis="y")
plt.show()
# Câu 6: Dự đoán giá đóng cửa phiên tiếp theo

df_ml = df.copy()

df_ml["Next_Close"] = df_ml["Close"].shift(-1)

df_ml = df_ml.dropna()

X = df_ml[["Open", "High", "Low", "Close", "Volume"]]
y = df_ml["Next_Close"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)
plt.figure(figsize=(14,6))
plt.plot(y_test.values, label="Giá thực tế")
plt.plot(y_pred, label="Giá dự đoán")
plt.title("So sánh giá thực tế và giá dự đoán")
plt.xlabel("Phiên giao dịch")
plt.ylabel("Giá đóng cửa")
plt.legend()
plt.grid(True)
plt.show()

latest_data = df[["Open", "High", "Low", "Close", "Volume"]].iloc[-1:]
next_price = model.predict(latest_data)

print("Giá đóng cửa FPT dự đoán cho phiên tiếp theo là:", next_price[0])
# Câu 7: Phân cụm các tháng có biến động giá tương tự nhau

monthly = df.groupby(["Year", "Month"]).agg({
    "Close": "mean",
    "Volume": "mean",
    "Daily_Return_%": "std"
}).reset_index()

monthly = monthly.rename(columns={
    "Close": "Avg_Close",
    "Volume": "Avg_Volume",
    "Daily_Return_%": "Volatility"
})

monthly = monthly.dropna()

X_cluster = monthly[["Avg_Close", "Avg_Volume", "Volatility"]]

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
monthly["Cluster"] = kmeans.fit_predict(X_cluster)

print(monthly[["Year", "Month", "Avg_Close", "Avg_Volume", "Volatility", "Cluster"]])
plt.figure(figsize=(10,6))
plt.scatter(monthly["Avg_Close"], monthly["Volatility"], c=monthly["Cluster"])
plt.title("Phân cụm các tháng theo giá trung bình và độ biến động")
plt.xlabel("Giá đóng cửa trung bình")
plt.ylabel("Độ biến động")
plt.grid(True)
plt.show()
# ===============================
# XUẤT RA FILE EXCEL
# ===============================

with pd.ExcelWriter("ket_qua_phan_tich_FPT.xlsx") as writer:
    df.to_excel(writer, sheet_name="Du_lieu_sau_xu_ly", index=False)
    gia_tb_nam.to_excel(writer, sheet_name="Gia_TB_Nam")
    gia_tb_thang.to_excel(writer, sheet_name="Gia_TB_Thang")
    monthly.to_excel(writer, sheet_name="Phan_cum_thang", index=False)

print("Đã xuất file ket_qua_phan_tich_FPT.xlsx")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1499 entries, 0 to 1498
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Ngày        1499 non-null   object
 1   Lần cuối    1499 non-null   object
 2   Mở          1499 non-null   object
 3   Cao         1499 non-null   object
 4   Thấp        1499 non-null   object
 5   KL          1499 non-null   object
 6   % Thay đổi  1499 non-null   object
dtypes: object(7)
memory usage: 82.1+ KB
None
              Ngày  Lần cuối        Mở       Cao      Thấp     KL % Thay đổi
count         1499      1499      1499      1499      1499   1499       1499
unique        1499      1010       996      1010      1015    772        569
top     02/01/2020  50,697.2  49,284.4  50,322.0  48,246.9  1.78M      0.00%
freq             1         9         9         7         7     10         86
Ngày          0
Lần cuối      0
Mở            0
Cao           0
Thấp          0
KL            0
% Thay 

KeyError: 'Date'